# 04 - Fine-tuning loss curves per latent scale

Train and validation curves of the LoRA-32 fine-tuning ([`experiments/03_autoencoder-fine-tuning-with-lora`](../experiments/03_autoencoder-fine-tuning-with-lora)),
one color per latent scale stage, from the `metrics.parquet` table of each `results/latent_scale_<n>`.

**Note:** runs after the next fine-tuning. The partial run's `metrics.parquet` has an older layout, so the cells below have no output yet.

In [ ]:
%matplotlib inline
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import polars as pl

from coolfrog.lib import plots
from coolfrog.lib.config import Grid
from coolfrog.lib.experiment import PATH_EXPERIMENTS, markdown_table, pull_experiment
from coolfrog.lib.huggingface import HuggingFaceHub
from coolfrog.lib.metrics import aggregate

PATH_EXPERIMENT = PATH_EXPERIMENTS / "03_autoencoder-fine-tuning-with-lora"
PATH_RESULTS = PATH_EXPERIMENT / "results"
plt.rcParams.update(plots.STYLE | {"figure.dpi": 110})

pull_experiment(HuggingFaceHub(), PATH_EXPERIMENT)  # tracker tables at the Hugging Face commit meta.json pins
STAGES = sorted(int(folder.name.removeprefix("latent_scale_")) for folder in PATH_RESULTS.glob("latent_scale_*"))


def stage_tables(table: str) -> pl.DataFrame:
    """`table` of every latent scale stage, `run` = its latent scale and grid, e.g. `latent scale 2 · 800x160x32`."""
    return pl.concat(
        pl.read_parquet(PATH_RESULTS / f"latent_scale_{latent_scale}" / f"{table}.parquet").with_columns(
            run=pl.lit(f"latent scale {latent_scale} · {'x'.join(map(str, Grid(latent_scale=latent_scale).shape))}")
        )
        for latent_scale in STAGES
    )


metrics = stage_tables(plots.METRICS)
plots.training_curves(metrics)

## Validation per channel and cost

Median over the validation designs per update, metric and channel. Update 0 is zero-shot, LoRA starts as the identity.

In [ ]:
display(plots.validation_curves(metrics))

validation = metrics.filter(pl.col("split") == plots.VALIDATION)
nrmse = (  # median over designs, then mean over channels, per run and update
    aggregate(validation, ["run", "update", "channel"], ["nrmse"]).group_by("run", "update").agg(pl.col("nrmse/median").mean()).sort("run", "update")
)
training_hours = (  # optimizer time without validation, every update counted once
    metrics.filter(pl.col("split") == plots.TRAIN)
    .group_by("run", "update")
    .agg(pl.col("seconds_per_update").first())
    .group_by("run")
    .agg((pl.col("seconds_per_update").sum() / 3600).alias("training time [h]"))
)
summary = (
    nrmse.group_by("run", maintain_order=True)
    .agg(pl.col("nrmse/median").first().alias("zero-shot NRMSE [%]"), pl.col("nrmse/median").last().alias("final NRMSE [%]"))
    .join(training_hours, on="run")
)
display(Markdown(markdown_table(summary, significant_digits=4)))